# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RayyanA24/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*
## Distributions

I examined the distributions of the main numerical features used in this project. Features such as impressions and search volume show a right-skewed distribution, where a small number of pages have much larger values than most others. This suggests that simple averages should be interpreted carefully.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git","clone","--depth","1",REPO_URL,REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

cols = [
    "search_volume",
    "impressions_90d",
    "avg_position",
    "ctr"
]

print(df[cols].describe())


       search_volume  impressions_90d  avg_position           ctr
count   27532.000000     30000.000000   30000.00000  30000.000000
mean      158.882391      5200.366300      16.34238      0.510733
std      1518.270825     16838.019547      15.21679      3.279162
min         0.000000         1.000000       0.00000      0.000000
25%         0.000000        81.000000       6.20000      0.000000
50%        10.000000       731.000000      10.80000      0.070000
75%        20.000000      3615.250000      22.30000      0.290000
max     74000.000000    517715.000000     245.00000    100.000000


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*
## Signal Tests

### Signal 1
Higher search volume is associated with higher impressions.

**Verdict:** MIXED

### Signal 2
Pages with lower average positions tend to receive fewer clicks.

**Verdict:** CONFIRMED

### Signal 3
Older content is more likely to require refreshing.

**Verdict:** MIXED

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Correlation: Search Volume vs Impressions")
print(
    df["search_volume"].corr(df["impressions_90d"])
)

print("\nCorrelation: CTR vs Average Position")
print(
    df["ctr"].corr(df["avg_position"])
)

print("\nMedian content age by trend")
print(
    df.groupby("trend_direction")["content_age_days"].median()
)

Correlation: Search Volume vs Impressions
0.0012030310696208905

Correlation: CTR vs Average Position
-0.07259029976832465

Median content age by trend
trend_direction
down      216.0
flat      231.0
new       279.0
stable    300.0
up        291.5
Name: content_age_days, dtype: float64


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*
## Flag-linked Test

One common refresh signal is pages that have high impressions but have not been updated recently. The data partially supports this assumption, although not every old page shows declining performance. This suggests the signal is useful but should not be used alone.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
high_imp = df["impressions_90d"] > df["impressions_90d"].median()
old_content = df["days_since_last_update"] > df["days_since_last_update"].median()

print(
    df.loc[
        high_imp & old_content,
        ["impressions_90d", "days_since_last_update"]
    ].head(10)
)


    impressions_90d  days_since_last_update
1             15320                      25
3             11751                      22
7              1724                      22
9              1240                     104
10            20919                     104
12             7228                     104
16            13848                     104
17             9449                      22
24            29541                     104
27             1197                     104


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*
## Practical Meaning

The observed signals can help prioritize pages for manual review, but they are not perfect predictors of declining content. A content team should treat these signals as decision-support rather than automatic decisions and combine them with human judgment.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.